# Chapter 6: Vector Search in Practice

The methods of the previous demo reach applications packaged inside libraries, search
engines, and databases. This demo follows book section 6.5 and sends one realistic query
through three of them: the FAISS library, the OpenSearch engine, and the pgvector
extension for PostgreSQL. (Lucene, the fourth system of the book section, gets its own
Java notebook.) A user types:

> a mind-bending heist thriller with Leonardo DiCaprio, released after 2005, rated 7.5 or better

No single retrieval method covers this request:

| Part of the query | Kind of match | Handled by |
|---|---|---|
| "mind-bending heist thriller" | meaning of the plot | vector search on the overview embedding |
| "with Leonardo DiCaprio" | exact name | BM25 on the actors field |
| "thriller" | category term | BM25 on the genres field |
| "released after 2005", "rated 7.5 or better" | hard constraints | filters on year and rating |

We first run the vector part in each system and then add the filters and the keyword
search.

**Prerequisites:** ch06-02 (IVF, PQ, HNSW), ch04 (BM25, the movies table), ch05-05 (hybrid
search with reciprocal rank fusion).

<div style="border-left: 4px solid #D97706; background: rgba(217, 119, 6, 0.06); padding: 0.6em 0.9em; margin: 0.6em 0; border-radius: 4px;">
<strong style="color:#D97706; text-transform:uppercase; font-size:0.78em; letter-spacing:0.06em;">Requires two Docker containers</strong><br>
pgvector and OpenSearch run as local services. Start both from the <code>demos/</code> folder with <code>docker compose -f docker/vector-search.yml up -d</code> (Docker Desktop must be running; the first start downloads about 3.5 GB of images). Stop them afterwards with <code>docker compose -f docker/vector-search.yml down</code>. The FAISS parts run without Docker.
</div>

## Setup

In [1]:
import io
import json
import time
from pathlib import Path

import faiss
import numpy as np
import psycopg2
from opensearchpy import OpenSearch, helpers
from shared.display import print_table, display_md
from shared.retrieval import document_frequencies, rank_collection_bm25
from shared.text import tokenize

## The movie collection

We use the 10,000 most-voted movies of the ch04 movies table, with their title, year,
rating, main actors, genres, and overview. Every overview was embedded once with
Qwen3-Embedding-0.6B, the model of ch05-05: 1,024 dimensions, unit length. The query
embeddings ship with the collection as well, so this notebook needs no embedding model.
The build script is <a href="eval/ch06_movies/make_movies.py">eval/ch06_movies/make_movies.py</a>.

In [2]:
DATA = Path("eval/ch06_movies")
movies = [json.loads(line) for line in (DATA / "movies.jsonl").open(encoding="utf-8")]
emb = np.load(DATA / "embeddings.npz")
X = emb["docs"].astype(np.float32)
faiss.normalize_L2(X)                          # unit length: inner product = cosine
QUERIES = {text: vec / np.linalg.norm(vec)
           for text, vec in zip(emb["query_texts"], emb["queries"].astype(np.float32))}

QUERY_TEXT = "a mind-bending heist thriller"
q = QUERIES[QUERY_TEXT]

def title(i):
    return f"{movies[i]['title']} ({movies[i]['year']})"

inception = next(m for m in movies if m["title"] == "Inception")
print_table(
    [[k, ", ".join(v) if isinstance(v, list) else v]
     for k, v in inception.items() if k not in ("imdb_id", "votes")],
    headers=["Field", "Inception"],
)
display_md(f"**Collection:** {len(movies):,} movies, embeddings {X.shape[0]:,} x {X.shape[1]}")

| Field    | Inception                                                                                                                                                                                                                                                                                |
|:---------|:-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| id       | 0                                                                                                                                                                                                                                                                                        |
| title    | Inception                                                                                                                                                                                                                                                                                |
| year     | 2010                                                                                                                                                                                                                                                                                     |
| rating   | 8.1                                                                                                                                                                                                                                                                                      |
| genres   | Action, Thriller, Science Fiction, Mystery, Adventure                                                                                                                                                                                                                                    |
| actors   | Leonardo DiCaprio, Joseph Gordon-Levitt, Ellen Page, Tom Hardy, Cillian Murphy                                                                                                                                                                                                           |
| overview | Cobb, a skilled thief who commits corporate espionage by infiltrating the subconscious of his targets is offered a chance to regain his old life as payment for a task considered to be impossible: "inception", the implantation of another person's idea into a target's subconscious. |

**Collection:** 10,000 movies, embeddings 10,000 x 1024

Before any index, we compute the exact answer for the vector part of the query: the
cosine between the query embedding and every overview.

In [3]:
exact_scores = X @ q
exact_order = np.argsort(-exact_scores)
rank_of = {int(i): r + 1 for r, i in enumerate(exact_order)}

print_table(
    [[r + 1, title(i), f"{exact_scores[i]:.3f}"] for r, i in enumerate(exact_order[:10])],
    headers=["Rank", "Movie", "Cosine"],
)
display_md(
    f"**Inception:** rank {rank_of[inception['id']]} with cosine {exact_scores[inception['id']]:.3f}  \n"
    f"Does its overview contain \"heist\" or \"mind-bending\"? "
    f"{any(w in inception['overview'].lower() for w in ('heist', 'mind-bending'))}"
)

|   Rank | Movie                      |   Cosine |
|-------:|:---------------------------|---------:|
|      1 | Parker (2013)              |    0.598 |
|      2 | Marnie (1964)              |    0.566 |
|      3 | Sneakers (1992)            |    0.564 |
|      4 | Marauders (2016)           |    0.56  |
|      5 | Shimmer Lake (2017)        |    0.56  |
|      6 | The Collector (2009)       |    0.554 |
|      7 | Hellraiser: Inferno (2000) |    0.552 |
|      8 | Man on a Ledge (2012)      |    0.551 |
|      9 | Lost Highway (1997)        |    0.55  |
|     10 | Now You See Me (2013)      |    0.548 |

**Inception:** rank 138 with cosine 0.482  
Does its overview contain "heist" or "mind-bending"? False

The vector part alone does not find Inception. Its overview speaks of a thief, corporate
espionage, and the subconscious, and other movies match "heist thriller" more literally.
The embedding understands the plot, but "mind-bending heist thriller" describes dozens of
movies equally well. The rest of the query has to do the work: the filters and the actor's
name. We first build the vector part in each system, then add the rest.

# Vector Search in Three Systems

## FAISS: a library of vector indexes

FAISS stores only vectors and their integer ids. The titles, actors, and years stay in the
application, here the `movies` list. We build the index of the book section: an inverted
file with 256 lists and PQ codes of 64 bytes (lever 3 and lever 2 of ch06-02).

In [4]:
index = faiss.index_factory(X.shape[1], "IVF256,PQ64", faiss.METRIC_INNER_PRODUCT)
start = time.perf_counter()
index.train(X)                      # k-means for the 256 centers and the PQ codebooks
index.add_with_ids(X, np.array([m["id"] for m in movies], dtype=np.int64))
build_s = time.perf_counter() - start

index.nprobe = 16                   # probe 16 of the 256 lists
scores, ids = index.search(q.reshape(1, -1), 10)

def overlap(found, k=10):
    return len(set(map(int, found[:k])) & set(map(int, exact_order[:k])))

print_table(
    [[r + 1, title(i), f"{s:.3f}", rank_of[int(i)]] for r, (i, s) in enumerate(zip(ids[0], scores[0]))],
    headers=["Rank", "Movie", "Approximate score", "Exact rank"],
)
display_md(f"**Build:** {build_s:.1f} s  \n**Recall@10 against exact search:** {overlap(ids[0]) / 10:.1f}  \n"
           f"**Memory:** {X.shape[1] * 4:,} bytes per vector as floats, 64 bytes as PQ codes")

|   Rank | Movie                      |   Approximate score |   Exact rank |
|-------:|:---------------------------|--------------------:|-------------:|
|      1 | Parker (2013)              |               0.55  |            1 |
|      2 | Riders (2002)              |               0.526 |           44 |
|      3 | Mission: Impossible (1996) |               0.523 |           14 |
|      4 | 12 Rounds (2009)           |               0.522 |           28 |
|      5 | Baby Driver (2017)         |               0.522 |           11 |
|      6 | Goldfinger (1964)          |               0.52  |           20 |
|      7 | The Collector (2009)       |               0.52  |            6 |
|      8 | Foolproof (2003)           |               0.52  |           18 |
|      9 | Lost Highway (1997)        |               0.519 |            9 |
|     10 | Cypher (2002)              |               0.518 |           43 |

**Build:** 21.2 s  
**Recall@10 against exact search:** 0.3  
**Memory:** 4,096 bytes per vector as floats, 64 bytes as PQ codes

The scores are approximate (PQ distances), and some exact top 10 movies are missing.
Changing the index string changes the method without changing the rest of the code. With
`RFlat`, FAISS reranks the best candidates with the exact vectors:

In [5]:
refined = faiss.index_factory(X.shape[1], "IVF256,PQ64,RFlat", faiss.METRIC_INNER_PRODUCT)
refined.train(X)
refined.add(X)
params = faiss.ParameterSpace()
params.set_index_parameter(refined, "nprobe", 16)
params.set_index_parameter(refined, "k_factor_rf", 10)     # rerank the best 100 exactly
scores_r, ids_r = refined.search(q.reshape(1, -1), 10)
display_md(f"**IVF256,PQ64,RFlat:** recall@10 = {overlap(ids_r[0]) / 10:.1f}, "
           f"top score {scores_r[0, 0]:.3f} (exact: {exact_scores[exact_order[0]]:.3f})")

**IVF256,PQ64,RFlat:** recall@10 = 0.9, top score 0.598 (exact: 0.598)

## OpenSearch: vector search as a field type

OpenSearch wraps Lucene in a distributed engine. Vector search is a field type of the
index mapping: `knn_vector` declares the dimensions, the similarity, and the ANN method.
We first check that the two Docker services are running.

In [6]:
def services_running():
    status = {}
    try:
        os_client = OpenSearch("http://localhost:9200", timeout=30)
        status["OpenSearch"] = os_client.info()["version"]["number"]
    except Exception as e:
        status["OpenSearch"] = f"not reachable ({type(e).__name__})"
    try:
        with psycopg2.connect(host="localhost", user="mmir", password="mmir", dbname="movies",
                              connect_timeout=5) as c, c.cursor() as cur:
            cur.execute("SELECT version()")
            status["PostgreSQL"] = cur.fetchone()[0].split(" on ")[0]
    except Exception as e:
        status["PostgreSQL"] = f"not reachable ({type(e).__name__})"
    return status

status = services_running()
print_table([[k, v] for k, v in status.items()], headers=["Service", "Status"])
if any("not reachable" in v for v in status.values()):
    raise RuntimeError("Start the services first: docker compose -f docker/vector-search.yml up -d")

| Service    | Status                                     |
|:-----------|:-------------------------------------------|
| OpenSearch | 3.9.0                                      |
| PostgreSQL | PostgreSQL 17.11 (Debian 17.11-1.pgdg12+2) |

The mapping follows the book: text fields for title, actors, and genres, numeric fields
for year and rating, and an HNSW graph (FAISS engine, $M = 16$) on the embedding.

In [7]:
client = OpenSearch("http://localhost:9200", timeout=60)
if client.indices.exists(index="movies"):
    client.indices.delete(index="movies")

client.indices.create(index="movies", body={
    "settings": {"index": {"knn": True, "number_of_shards": 1, "number_of_replicas": 0}},
    "mappings": {"properties": {
        "title":   {"type": "text"},
        "actors":  {"type": "text"},
        "genres":  {"type": "text"},
        "year":    {"type": "integer"},
        "rating":  {"type": "float"},
        "embedding": {
            "type": "knn_vector",
            "dimension": 1024,
            "space_type": "cosinesimil",
            "method": {"name": "hnsw", "engine": "faiss",
                       "parameters": {"m": 16, "ef_construction": 128}},
        },
    }},
})

start = time.perf_counter()
helpers.bulk(client, ({
    "_index": "movies", "_id": m["id"], "title": m["title"], "year": m["year"],
    "rating": m["rating"], "actors": ", ".join(m["actors"]), "genres": ", ".join(m["genres"]),
    "embedding": vec.tolist(),
} for m, vec in zip(movies, X)), chunk_size=500)
client.indices.refresh(index="movies")
display_md(f"**Indexed** {client.count(index='movies')['count']:,} movies in "
           f"{time.perf_counter() - start:.0f} s")

**Indexed** 10,000 movies in 9 s

The k-NN query names the vector field, the query vector, and $k$:

In [8]:
response = client.search(index="movies", body={
    "size": 10,
    "query": {"knn": {"embedding": {"vector": q.tolist(), "k": 10}}},
})
hits = response["hits"]["hits"]
print_table(
    [[r + 1, title(int(h["_id"])), f"{h['_score']:.4f}", f"{exact_scores[int(h['_id'])]:.4f}",
      f"{(1 + exact_scores[int(h['_id'])]) / 2:.4f}"] for r, h in enumerate(hits)],
    headers=["Rank", "Movie", "OpenSearch score", "Cosine", "(1 + cos) / 2"],
)
display_md(f"**Recall@10 against exact search:** {overlap([h['_id'] for h in hits]) / 10:.1f}")

|   Rank | Movie                      |   OpenSearch score |   Cosine |   (1 + cos) / 2 |
|-------:|:---------------------------|-------------------:|---------:|----------------:|
|      1 | Parker (2013)              |             0.7992 |   0.5985 |          0.7992 |
|      2 | Marnie (1964)              |             0.7832 |   0.5665 |          0.7832 |
|      3 | Sneakers (1992)            |             0.7821 |   0.5642 |          0.7821 |
|      4 | Marauders (2016)           |             0.7799 |   0.5598 |          0.7799 |
|      5 | Shimmer Lake (2017)        |             0.7799 |   0.5598 |          0.7799 |
|      6 | The Collector (2009)       |             0.777  |   0.554  |          0.777  |
|      7 | Hellraiser: Inferno (2000) |             0.7758 |   0.5516 |          0.7758 |
|      8 | Man on a Ledge (2012)      |             0.7753 |   0.5506 |          0.7753 |
|      9 | Lost Highway (1997)        |             0.7751 |   0.5502 |          0.7751 |
|     10 | Now You See Me (2013)      |             0.7742 |   0.5484 |          0.7742 |

**Recall@10 against exact search:** 1.0

HNSW finds the exact top 10 here. The score is not the cosine itself: like Lucene,
OpenSearch reports $(1 + \cos)/2$, so that scores are positive and higher is always
better.

## pgvector: vectors in SQL

pgvector adds a `vector` column type to PostgreSQL. We create the movies table with a
vector column, load the data with `COPY`, and add the full-text column of ch04-03 that we
need later for the keyword part.

In [9]:
con = psycopg2.connect(host="localhost", user="mmir", password="mmir", dbname="movies")
con.autocommit = True
cur = con.cursor()

def sql(statement, params=None):
    """Run a statement and return all rows (or None for statements without a result)."""
    cur.execute(statement, params)
    return cur.fetchall() if cur.description else None

def vec_literal(v):
    return "[" + ",".join(f"{x:.6f}" for x in v) + "]"

sql("CREATE EXTENSION IF NOT EXISTS vector")
sql("DROP TABLE IF EXISTS movies")
sql("""CREATE TABLE movies (
         id int PRIMARY KEY, title text, year int, rating real,
         actors text, genres text, overview text, embedding vector(1024))""")

clean = lambda s: s.replace("\\", " ").replace("\t", " ").replace("\n", " ")
buf = io.StringIO()
for m, vec in zip(movies, X):
    buf.write("\t".join([str(m["id"]), clean(m["title"]), str(m["year"]), str(m["rating"]),
                         clean(", ".join(m["actors"])), clean(", ".join(m["genres"])),
                         clean(m["overview"]), vec_literal(vec)]) + "\n")
buf.seek(0)
start = time.perf_counter()
cur.copy_from(buf, "movies", columns=("id", "title", "year", "rating", "actors", "genres",
                                      "overview", "embedding"))
sql("""ALTER TABLE movies ADD COLUMN tsv tsvector GENERATED ALWAYS AS (
         to_tsvector('english', title || ' ' || actors || ' ' || genres || ' ' || overview)) STORED""")
sql("CREATE INDEX ON movies USING GIN (tsv)")
sql("ANALYZE movies")
display_md(f"**Loaded** {sql('SELECT count(*) FROM movies')[0][0]:,} rows in "
           f"{time.perf_counter() - start:.0f} s")

**Loaded** 10,000 rows in 2 s

pgvector works with distances, so smaller is better: `<=>` is the cosine distance
$1 - \cos$. Without an index, the query below is an exact scan. Then we add the HNSW index
of the book and run the same query again.

In [10]:
q_lit = vec_literal(q)
KNN = "SELECT id, embedding <=> %s AS distance FROM movies ORDER BY embedding <=> %s LIMIT 10"

def plan(statement, params):
    """The access path PostgreSQL chooses: a sequential scan or one of the indexes."""
    lines = [row[0] for row in sql("EXPLAIN " + statement, params)]
    for line in lines:
        if "Index Scan using" in line:
            return "Index Scan using " + line.split("Index Scan using ")[1].split()[0]
    return "Seq Scan + Sort" if any("Seq Scan" in line for line in lines) else lines[0].strip()

exact_rows = sql(KNN, (q_lit, q_lit))
plan_scan = plan(KNN, (q_lit, q_lit))

start = time.perf_counter()
sql("CREATE INDEX movies_hnsw ON movies USING hnsw (embedding vector_cosine_ops) "
    "WITH (m = 16, ef_construction = 64)")
build_s = time.perf_counter() - start
sql("SET hnsw.ef_search = 100")
plan_default = plan(KNN, (q_lit, q_lit))
sql("SET enable_seqscan = off")         # make the planner use the index
hnsw_rows = sql(KNN, (q_lit, q_lit))
plan_forced = plan(KNN, (q_lit, q_lit))
sql("SET enable_seqscan = on")

print_table(
    [[r + 1, title(i), f"{d:.4f}", f"{1 - exact_scores[i]:.4f}"] for r, (i, d) in enumerate(hnsw_rows)],
    headers=["Rank", "Movie", "Distance <=> (HNSW)", "1 - cosine"],
)
display_md(
    f"- **Before the index:** {plan_scan}\n"
    f"- **After the index** (built in {build_s:.0f} s), planner's choice: {plan_default}\n"
    f"- **Index forced** (`enable_seqscan = off`): {plan_forced}\n"
    f"- **HNSW top 10 = exact top 10:** {[i for i, _ in hnsw_rows] == [i for i, _ in exact_rows]}"
)

|   Rank | Movie                      |   Distance <=> (HNSW) |   1 - cosine |
|-------:|:---------------------------|----------------------:|-------------:|
|      1 | Parker (2013)              |                0.4015 |       0.4015 |
|      2 | Marnie (1964)              |                0.4335 |       0.4335 |
|      3 | Sneakers (1992)            |                0.4358 |       0.4358 |
|      4 | Marauders (2016)           |                0.4402 |       0.4402 |
|      5 | Shimmer Lake (2017)        |                0.4402 |       0.4402 |
|      6 | The Collector (2009)       |                0.446  |       0.446  |
|      7 | Hellraiser: Inferno (2000) |                0.4484 |       0.4484 |
|      8 | Man on a Ledge (2012)      |                0.4494 |       0.4494 |
|      9 | Lost Highway (1997)        |                0.4498 |       0.4498 |
|     10 | Now You See Me (2013)      |                0.4516 |       0.4516 |

- **Before the index:** Seq Scan + Sort
- **After the index** (built in 1 s), planner's choice: Seq Scan + Sort
- **Index forced** (`enable_seqscan = off`): Index Scan using movies_hnsw
- **HNSW top 10 = exact top 10:** True

For the `ORDER BY ... LIMIT` pattern, PostgreSQL's planner can use the HNSW index, but it
decides by estimated cost. With only 10,000 rows, a sequential scan with a sort is cheap,
and the planner may keep it even after the index exists. With millions of rows, the index
wins. Forcing the index shows that HNSW returns the exact top 10 here, and that `<=>`
is exactly $1 - \cos$. The index
parameters ($m$, `ef_construction`) and the query parameter `hnsw.ef_search` are the HNSW
parameters of ch06-02.

pgvector also exposes the other levers. The book's example reduces every vector to its
sign bits for the candidate search and reranks the candidates with the full vectors, in
plain SQL:

In [11]:
sql("CREATE INDEX movies_bits ON movies USING hnsw "
    "((binary_quantize(embedding)::bit(1024)) bit_hamming_ops)")
BINARY_RERANK = """
    SELECT id FROM (
      SELECT id, embedding FROM movies
      ORDER BY binary_quantize(embedding)::bit(1024) <~> binary_quantize(%s::vector)
      LIMIT %s
    ) AS candidates
    ORDER BY embedding <=> %s
    LIMIT 10"""

rows = []
for k_prime in [10, 40, 100]:
    found = [r[0] for r in sql(BINARY_RERANK, (q_lit, k_prime, q_lit))]
    rows.append([k_prime, f"{overlap(found) / 10:.1f}"])
print_table(rows, headers=["Candidates k' from 128-byte bit codes", "Recall@10 after exact rerank"])

|   Candidates k' from 128-byte bit codes |   Recall@10 after exact rerank |
|----------------------------------------:|-------------------------------:|
|                                      10 |                            0.2 |
|                                      40 |                            0.6 |
|                                     100 |                            0.8 |

With 1 bit per component, a vector shrinks from 4,096 to 128 bytes. The Hamming ranking
alone is coarse, but the good matches are among the first 100 candidates, and the exact
rerank restores the top 10.

# Adding Filters and Keyword Search

## Three ways to filter

The filters "released after 2005" and "rated 7.5 or better" admit only a small part of
the collection:

In [12]:
admitted = np.array([m["id"] for m in movies if m["year"] > 2005 and m["rating"] >= 7.5])
display_md(f"**Admitted by the filter:** {len(admitted)} of {len(movies):,} movies "
           f"({len(admitted) / len(movies):.1%})")

**Admitted by the filter:** 261 of 10,000 movies (2.6%)

pgvector shows all three strategies of the book in one system. We run the same filtered
query three times: once as PostgreSQL plans it, once with the index forced and a plain
index scan (post-filtering), and once with the iterative index scan of pgvector 0.8.

In [13]:
FILTERED = ("SELECT id FROM movies WHERE year > 2005 AND rating >= 7.5 "
            "ORDER BY embedding <=> %s LIMIT 10")
sql("SET hnsw.ef_search = 40")      # the default

settings = {
    "planner's choice": ["SET enable_seqscan = on", "SET hnsw.iterative_scan = off"],
    "index forced, plain scan": ["SET enable_seqscan = off", "SET hnsw.iterative_scan = off"],
    "index forced, iterative scan": ["SET enable_seqscan = off", "SET hnsw.iterative_scan = relaxed_order"],
}
rows = []
for name, statements in settings.items():
    for s in statements:
        sql(s)
    found = [r[0] for r in sql(FILTERED, (q_lit,))]
    rows.append([name, plan(FILTERED, (q_lit,)), len(found),
                 ", ".join(title(i) for i in found[:3])])
sql("SET enable_seqscan = on")
sql("SET hnsw.iterative_scan = relaxed_order")
print_table(rows, headers=["Setting", "Plan", "Rows returned", "First results"])

| Setting                      | Plan                         |   Rows returned | First results                                        |
|:-----------------------------|:-----------------------------|----------------:|:-----------------------------------------------------|
| planner's choice             | Seq Scan + Sort              |              10 | River (2015), Inception (2010), The Lost Room (2006) |
| index forced, plain scan     | Index Scan using movies_hnsw |               0 |                                                      |
| index forced, iterative scan | Index Scan using movies_hnsw |              10 | River (2015), Inception (2010), The Lost Room (2006) |

- **Planner's choice: pre-filtering.** The filter is selective, so PostgreSQL skips the
  vector index, reads the admitted rows, and sorts them exactly. The result is exact.
- **Plain index scan: post-filtering.** The HNSW scan delivers about `ef_search` = 40
  candidates, and the filter removes most of them. The query asks for 10 rows and silently
  gets fewer, possibly none at all.
- **Iterative scan: a stream of candidates.** The index scan continues until enough rows
  pass the filter.

The filter alone already changes the result: Inception moves far up among the admitted
movies.

In [14]:
filtered_exact = [int(i) for i in exact_order if i in set(admitted)]
display_md(f"**Inception among the admitted movies:** rank "
           f"{filtered_exact.index(inception['id']) + 1} of {len(admitted)}  \n"
           f"**Top 5 admitted:** {', '.join(title(i) for i in filtered_exact[:5])}")

**Inception among the admitted movies:** rank 2 of 261  
**Top 5 admitted:** River (2015), Inception (2010), The Lost Room (2006), Kingsman: The Secret Service (2015), An Inspector Calls (2015)

## Keyword search and rank fusion

The actor's name is an exact match, which BM25 handles better than an embedding. As in
the hybrid search of ch05-05, we fuse the dense ranking and the BM25 ranking with
**reciprocal rank fusion** (RRF), which only needs ranks:

$$\text{RRF}(d) = \sum_{\text{rankings } r} \frac{1}{60 + \text{rank}_r(d)}$$

## FAISS: filter in the index, BM25 and fusion in the application

FAISS has no metadata. The application evaluates the filter and passes the admitted ids
to the search, which skips all other ids while it scans the probed lists (filtering during
the search). The keyword part needs a separate BM25 engine; we use the BM25 of ch04 on the
actors and genres of the admitted movies.

In [15]:
selector = faiss.IDSelectorBatch(admitted.astype(np.int64))
search_params = faiss.SearchParametersIVF(sel=selector, nprobe=32)
_, dense_ids = index.search(q.reshape(1, -1), 50, params=search_params)
dense_ids = [int(i) for i in dense_ids[0] if i >= 0]

fields = {m["id"]: tokenize(" ".join(m["actors"] + m["genres"])) for m in movies}
corpus = {i: fields[i] for i in admitted.tolist()}
bm25 = rank_collection_bm25(tokenize("Leonardo DiCaprio thriller"), corpus,
                            document_frequencies(corpus))
sparse_ids = [i for i, s in bm25 if s > 0][:50]

def rrf(*rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)

faiss_hybrid = rrf(dense_ids, sparse_ids)[:10]
print_table(
    [[r + 1, title(i), dense_ids.index(i) + 1 if i in dense_ids else "", 
      sparse_ids.index(i) + 1 if i in sparse_ids else "", ", ".join(movies[i]["actors"][:2])]
     for r, i in enumerate(faiss_hybrid)],
    headers=["Rank", "Movie", "Dense rank", "BM25 rank", "Main actors"],
)

|   Rank | Movie                     |   Dense rank |   BM25 rank | Main actors                             |
|-------:|:--------------------------|-------------:|------------:|:----------------------------------------|
|      1 | Inception (2010)          |            3 |           4 | Leonardo DiCaprio, Joseph Gordon-Levitt |
|      2 | The Departed (2006)       |           12 |           2 | Leonardo DiCaprio, Matt Damon           |
|      3 | Who Am I (2014)           |           11 |           9 | Wotan Wilke Möhring                     |
|      4 | Django Unchained (2012)   |           20 |           5 | Jamie Foxx, Christoph Waltz             |
|      5 | An Inspector Calls (2015) |            4 |          22 | David Thewlis, Miranda Richardson       |
|      6 | The Raid 2 (2014)         |           14 |          14 | Ken'ichi Endô                           |
|      7 | Ghost Recon: Alpha (2012) |           10 |          19 | Charlotte Rampling                      |
|      8 | I Saw the Devil (2010)    |           18 |          12 | Lee Byung-hun                           |
|      9 | The Chaser (2008)         |           15 |          16 | Jeong In-gi                             |
|     10 | V for Vendetta (2006)     |            7 |          29 | Natalie Portman, Hugo Weaving           |

The two rankings agree on Inception: it is near the top of the filtered dense ranking and
among the DiCaprio movies of the BM25 ranking, so RRF puts it first. Movies found by only
one ranking follow.

## OpenSearch: filter inside the query, fusion in a search pipeline

OpenSearch passes a filter inside the query to the engine, which filters during the
search. For the keyword part, a `hybrid` query runs its sub-queries separately, and a
search pipeline fuses their results. Since version 2.19, the `score-ranker-processor`
implements RRF.

In [16]:
client.transport.perform_request("PUT", "/_search/pipeline/rrf-pipeline", body={
    "phase_results_processors": [{"score-ranker-processor": {
        "combination": {"technique": "rrf", "rank_constant": 60}}}],
})

filters = {"bool": {"filter": [
    {"range": {"year": {"gt": 2005}}},
    {"range": {"rating": {"gte": 7.5}}},
]}}
response = client.search(index="movies", params={"search_pipeline": "rrf-pipeline"}, body={
    "size": 10,
    "query": {"hybrid": {
        "queries": [
            {"bool": {"should": [
                {"match_phrase": {"actors": "Leonardo DiCaprio"}},
                {"match": {"genres": "thriller"}},
            ]}},
            {"knn": {"embedding": {"vector": q.tolist(), "k": 50}}},
        ],
        "filter": filters,
    }},
})
opensearch_hybrid = [int(h["_id"]) for h in response["hits"]["hits"]]
print_table(
    [[r + 1, title(int(h["_id"])), f"{h['_score']:.4f}"] for r, h in enumerate(response["hits"]["hits"])],
    headers=["Rank", "Movie", "RRF score"],
)

|   Rank | Movie                               |   RRF score |
|-------:|:------------------------------------|------------:|
|      1 | Inception (2010)                    |      0.0318 |
|      2 | The Wolf of Wall Street (2013)      |      0.0301 |
|      3 | Who Am I (2014)                     |      0.0292 |
|      4 | River (2015)                        |      0.0164 |
|      5 | Before the Flood (2016)             |      0.0164 |
|      6 | Shutter Island (2010)               |      0.0161 |
|      7 | The Lost Room (2006)                |      0.0159 |
|      8 | The Departed (2006)                 |      0.0159 |
|      9 | Kingsman: The Secret Service (2015) |      0.0156 |
|     10 | An Inspector Calls (2015)           |      0.0154 |

A movie at rank 1 in both sub-queries would score $2/61 \approx 0.0328$, the maximum.
Inception comes close because it ranks high in both; movies found by only one sub-query
score at most $1/61 \approx 0.0164$. The `filter` of the hybrid query applies to both
sub-queries.

## pgvector: WHERE clauses and RRF in SQL

In SQL, the filter is a `WHERE` clause, and the iterative scan (set above) keeps the HNSW
scan going until enough rows pass. The keyword part uses PostgreSQL's full-text search,
and the fusion is a `FULL OUTER JOIN` of two ranked lists. This is the query of the book:

In [17]:
HYBRID = """
WITH dense AS (
  SELECT id, ROW_NUMBER() OVER (ORDER BY embedding <=> %(q)s) AS r
  FROM movies
  WHERE year > 2005 AND rating >= 7.5
  ORDER BY embedding <=> %(q)s
  LIMIT 50
),
sparse AS (
  SELECT id, ROW_NUMBER() OVER (ORDER BY ts_rank(tsv, query) DESC) AS r
  FROM movies, to_tsquery('english', '(leonardo <-> dicaprio) | thriller') AS query
  WHERE tsv @@ query AND year > 2005 AND rating >= 7.5
  ORDER BY ts_rank(tsv, query) DESC
  LIMIT 50
)
SELECT COALESCE(d.id, s.id) AS id, d.r AS dense_rank, s.r AS sparse_rank,
       COALESCE(1.0 / (60 + d.r), 0) + COALESCE(1.0 / (60 + s.r), 0) AS rrf
FROM dense d
FULL OUTER JOIN sparse s ON d.id = s.id
ORDER BY rrf DESC
LIMIT 10"""

rows = sql(HYBRID, {"q": q_lit})
pgvector_hybrid = [r[0] for r in rows]
print_table(
    [[k + 1, title(i), d or "", s or "", f"{float(f):.4f}"] for k, (i, d, s, f) in enumerate(rows)],
    headers=["Rank", "Movie", "Dense rank", "Text rank", "RRF score"],
)

|   Rank | Movie                          |   Dense rank |   Text rank |   RRF score |
|-------:|:-------------------------------|-------------:|------------:|------------:|
|      1 | Inception (2010)               |            2 |           3 |      0.032  |
|      2 | The Wolf of Wall Street (2013) |            8 |           5 |      0.0301 |
|      3 | The Departed (2006)            |           17 |           1 |      0.0294 |
|      4 | No Country for Old Men (2007)  |            7 |          11 |      0.029  |
|      5 | Shutter Island (2010)          |           32 |           2 |      0.027  |
|      6 | The Raid 2 (2014)              |           12 |          18 |      0.0267 |
|      7 | Who Am I (2014)                |           10 |          22 |      0.0265 |
|      8 | V for Vendetta (2006)          |           31 |           7 |      0.0259 |
|      9 | An Inspector Calls (2015)      |            5 |          35 |      0.0259 |
|     10 | Paprika (2006)                 |           13 |          23 |      0.0257 |

`ts_rank` is not BM25, and the text index covers title, overview, actors, and genres. RRF
only uses the order of the text ranking, which keeps the difference small.

## Comparing the three systems

In [18]:
results = {"FAISS + app BM25": faiss_hybrid, "OpenSearch hybrid": opensearch_hybrid,
           "pgvector SQL": pgvector_hybrid}
print_table(
    [[r + 1] + [title(res[r]) if r < len(res) else "" for res in results.values()] for r in range(10)],
    headers=["Rank"] + list(results),
)
display_md("**Inception's rank:** " + ", ".join(
    f"{name}: {res.index(inception['id']) + 1 if inception['id'] in res else 'not in top 10'}"
    for name, res in results.items()))

|   Rank | FAISS + app BM25          | OpenSearch hybrid                   | pgvector SQL                   |
|-------:|:--------------------------|:------------------------------------|:-------------------------------|
|      1 | Inception (2010)          | Inception (2010)                    | Inception (2010)               |
|      2 | The Departed (2006)       | The Wolf of Wall Street (2013)      | The Wolf of Wall Street (2013) |
|      3 | Who Am I (2014)           | Who Am I (2014)                     | The Departed (2006)            |
|      4 | Django Unchained (2012)   | River (2015)                        | No Country for Old Men (2007)  |
|      5 | An Inspector Calls (2015) | Before the Flood (2016)             | Shutter Island (2010)          |
|      6 | The Raid 2 (2014)         | Shutter Island (2010)               | The Raid 2 (2014)              |
|      7 | Ghost Recon: Alpha (2012) | The Lost Room (2006)                | Who Am I (2014)                |
|      8 | I Saw the Devil (2010)    | The Departed (2006)                 | V for Vendetta (2006)          |
|      9 | The Chaser (2008)         | Kingsman: The Secret Service (2015) | An Inspector Calls (2015)      |
|     10 | V for Vendetta (2006)     | An Inspector Calls (2015)           | Paprika (2006)                 |

**Inception's rank:** FAISS + app BM25: 1, OpenSearch hybrid: 1, pgvector SQL: 1

All three systems agree on Inception, but below it the rankings differ: The Departed is
second in FAISS and eighth in OpenSearch, and The Wolf of Wall Street does not appear in
the FAISS top 10 at all. RRF only sees ranks, so we look at the two input rankings each
system fused. For OpenSearch, we run its two sub-queries on their own.

In [19]:
os_filter = filters["bool"]["filter"]
os_knn = client.search(index="movies", body={"size": 50, "query": {"knn": {"embedding": {
    "vector": q.tolist(), "k": 50, "filter": filters}}}})
os_bm25 = client.search(index="movies", body={"size": 50, "query": {"bool": {
    "should": [{"match_phrase": {"actors": "Leonardo DiCaprio"}}, {"match": {"genres": "thriller"}}],
    "filter": os_filter}}})
pg_ranks = {r[0]: (r[1], r[2]) for r in sql(HYBRID.replace("LIMIT 10", "LIMIT 100"), {"q": q_lit})}

ids_of = lambda resp: [int(h["_id"]) for h in resp["hits"]["hits"]]
rank_in = lambda ranking, i: ranking.index(i) + 1 if i in ranking else "missing"
rows = []
for name in ["Inception", "The Departed", "The Wolf of Wall Street"]:
    i = next(m["id"] for m in movies if m["title"] == name)
    pg_dense, pg_text = pg_ranks.get(i, ("missing", "missing"))
    rows.append([name, rank_in(filtered_exact, i),
                 f"{rank_in(dense_ids, i)} / {rank_in(sparse_ids, i)}",
                 f"{rank_in(ids_of(os_knn), i)} / {rank_in(ids_of(os_bm25), i)}",
                 f"{pg_dense or 'missing'} / {pg_text or 'missing'}"])
print_table(rows, headers=["Movie", "Exact dense (filtered)", "FAISS dense / BM25",
                           "OpenSearch k-NN / BM25", "pgvector dense / text"])
display_md(f"**Admitted movies FAISS can see** in its {search_params.nprobe} probed lists "
           f"of 256: {len(dense_ids)} of {len(admitted)}")

| Movie                   |   Exact dense (filtered) | FAISS dense / BM25   | OpenSearch k-NN / BM25   | pgvector dense / text   |
|:------------------------|-------------------------:|:---------------------|:-------------------------|:------------------------|
| Inception               |                        2 | 3 / 4                | 2 / 4                    | 2 / 3                   |
| The Departed            |                       17 | 12 / 2               | 17 / 3                   | 17 / 1                  |
| The Wolf of Wall Street |                        8 | missing / 6          | 8 / 5                    | 8 / 5                   |

**Admitted movies FAISS can see** in its 32 probed lists of 256: 26 of 261

Three implementation choices explain the differences:

- **The approximate dense search.** pgvector's planner scans the 261 admitted rows exactly.
  OpenSearch's HNSW, filtered during the graph search, matches the exact ranks here. FAISS
  filters during an IVF search with PQ codes: the 32 probed lists hold only a fraction of
  the admitted movies, so The Wolf of Wall Street is never scored, and The Departed moves
  up because better candidates are missing too. This is the selective-filter problem of the
  book: raise $n_{\text{probe}}$ or switch to an exact scan over the admitted vectors.
- **The keyword ranking.** Our BM25 scores actors and genres as one field. OpenSearch scores
  the phrase on actors and the term on genres separately and adds the two scores; its
  length normalization favors movies with short actor lists. PostgreSQL's `ts_rank` covers
  title, overview, actors, and genres and has no IDF. All three find the same DiCaprio
  movies, in different orders.
- **The fusion depth.** FAISS and pgvector fuse the top 50 of both rankings. OpenSearch's
  hybrid query fuses only the top `size` = 10 hits of each sub-query, so The Departed, rank
  17 in k-NN, gets no dense contribution there.

The lesson: approximate search is not one result but a family of results. The method (IVF
with PQ, HNSW, an exact scan chosen by the planner), its parameters, the keyword scorer,
and the fusion depth all shift the ranking below the clear winner. Comparing systems, or
tuning one, therefore needs measurements against an exact or judged reference, as in
ch06-02 and ch06-03.

<div style="border-left: 4px solid #C8102E; background: rgba(200, 16, 46, 0.06); padding: 0.6em 0.9em; margin: 0.6em 0; border-radius: 4px;">
<strong style="color:#C8102E; text-transform:uppercase; font-size:0.78em; letter-spacing:0.06em;">Meaning, names, and constraints need three ingredients</strong><br>
The vector search alone ranks Inception far down: "mind-bending heist thriller" fits many plots. The filters remove every movie that cannot qualify, and BM25 matches the actor's name exactly. Fused with RRF, the three ingredients put Inception first in all three systems. Below the winner, the rankings differ with each system's approximate search, keyword scorer, and fusion depth. The systems also differ in how much of the combination they do for us: FAISS leaves filters, keywords, and fusion to the application; OpenSearch runs everything in one query; pgvector expresses it in SQL next to the relational data.
</div>

## Choosing a system

| | FAISS | OpenSearch | pgvector |
|---|---|---|---|
| What it stores | vectors and integer ids | documents with text, numbers, vectors | rows with any columns, including vectors |
| ANN methods | all of ch06-02, freely combined | HNSW, IVF, PQ (FAISS or Lucene engine) | HNSW, IVFFlat, bit and half-precision codes |
| Filters | id selector from the application | inside the k-NN query | `WHERE`, iterative index scans |
| Keyword search and fusion | in the application | `hybrid` query with RRF pipeline | full-text search, RRF in SQL |
| Suits | maximum control and speed, GPUs | large, distributed search applications | collections next to relational data on one server |

## Summary

- One realistic query needs vector search for the meaning, BM25 for names and terms, and
  filters for hard constraints. The vector part alone ranked Inception only far down.
- FAISS, OpenSearch, and pgvector implement the same ANN methods. Their scores follow
  different conventions: approximate inner products, $(1 + \cos)/2$, and the cosine
  distance $1 - \cos$.
- Filters can run before, after, or during the ANN search. Post-filtering can silently
  return too few results; pre-filtering and iterative scans avoid that.
- Reciprocal rank fusion combines dense and keyword rankings without calibrating their
  scores.

## Try it yourself

**1. Other queries.** The collection ships embeddings for a few more queries. Run the
exact vector search for each and look at the top 3:

In [20]:
for text, vec in QUERIES.items():
    top = np.argsort(-(X @ vec))[:3]
    display_md(f"**{text}:** " + "; ".join(title(i) for i in top))

**a mind-bending heist thriller:** Parker (2013); Marnie (1964); Sneakers (1992)

**a lonely robot falls in love:** Enthiran (2010); EVA (2011); Short Circuit (1986)

**astronauts travel through a wormhole to save humanity:** Interstellar (2014); Lifeforce (1985); Gravity (2013)

**a con artist pretends to be a pilot and a doctor:** Just Go with It (2011); An Honest Liar (2014); Catch Me If You Can (2002)

**a shark terrorizes a beach town:** Jaws (1975); Shark Night (2011); Dark Tide (2012)

**2. Your own query.** With `sentence-transformers` installed, you can embed new queries
(the first call downloads the model, about 1.2 GB):

In [21]:
def embed(text):
    from sentence_transformers import SentenceTransformer
    global _model
    if "_model" not in globals():
        _model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")
    return _model.encode(text, prompt_name="query", normalize_embeddings=True).astype(np.float32)

# Example (uncomment to run): q = embed("a dream within a dream")

**3. A milder filter.** Change the filter to `year > 1990` (no rating condition) and rerun
the three pgvector settings. Which strategy does the planner choose now, and does the plain
index scan still return too few rows?

## Cleanup

In [22]:
cur.close()
con.close()
client.indices.delete(index="movies")
display_md("**Closed** the PostgreSQL connection and deleted the OpenSearch index. "
           "Stop the containers with `docker compose -f docker/vector-search.yml down`.")

**Closed** the PostgreSQL connection and deleted the OpenSearch index. Stop the containers with `docker compose -f docker/vector-search.yml down`.